## Assignment 3: Sequence Models and Transformers

### Assignment Overview

This assignment covers the transition from classical probabilistic sequence models (Naive Bayes, HMMs, CRFs) to modern deep learning architectures. You will explore Recurrent Neural Networks (LSTMs), the attention mechanism, the Transformer architecture, and pre-trained models such as BERT.

You are required to complete six coding-related tasks. For each task, you will be working with specified data or parameters. Please submit your solutions in this single Jupyter Notebook (`.ipynb`) file **without editing the questions, or zipping the files into .zip or .rar files**, clearly marking each task. Ensure your code is well-commented and your findings are explained in markdown cells where requested.

### Task 1: Sequence Classification with Naive Bayes (15 Marks)

**Objective:** To implement a sequence classification model using Multinomial Naive Bayes, which applies the independence assumption to sequence data.

**Description:** Naive Bayes is a fundamental probabilistic classifier that assumes feature independence. For text sequences, this means treating each word position independently, which is a simplification but often effective for classification tasks.

**Your task is to:**

1.  **Load and Preprocess Data:** Use the `sample_reviews` and `sample_labels` provided in the code cell below. You will need to tokenize the text and build a vocabulary.
2.  **Implement Naive Bayes Training:** Create a function `train_naive_bayes(texts, labels)` that:
    * Computes the prior probabilities for each class (P(class)).
    * Computes the conditional probabilities P(word|class) for each word in the vocabulary given each class.
    * Uses Laplace (Add-One) smoothing to handle unseen words.
    * Returns the prior probabilities, conditional probabilities, and vocabulary.
3.  **Implement Prediction:** Create a function `predict_naive_bayes(text, priors, conditionals, vocab)` that:
    * Takes a text sequence, prior probabilities, conditional probabilities, and vocabulary.
    * Computes the posterior probability for each class using Bayes' rule: P(class|text) ∝ P(class) × ∏ P(word|class).
    * Returns the predicted class and the probabilities for both classes.
4.  **Evaluate:** Train on the provided data and predict the class for each review. Print the predictions and the probability scores.

In [1]:
# Your code for Task 1 here
from collections import Counter, defaultdict
import numpy as np
import math

# Data for Task 1
sample_reviews = [
    # Positive reviews
    "this movie was fantastic and i really loved it",
    "an amazing performance by the main actor",
    "wonderful storyline and great cinematography",
    "i enjoyed every minute of this film",
    "the film was outstanding and very entertaining",
    "incredible acting and a moving plot",
    "the best movie I have seen this year",
    "brilliant directing and superb special effects",
    "the soundtrack was beautiful and fitting",
    "characters were relatable and the pacing was perfect",
    # Negative reviews
    "i hated this film it was awful and boring",
    "a complete waste of time do not watch",
    "terrible script and wooden acting",
    "the movie was badly edited and confusing",
    "plot holes everywhere and poor dialogue",
    "i was disappointed and almost fell asleep",
    "this movie is not worth your time",
    "boring and predictable from beginning to end",
    "the worst film experience I’ve ever had",
    "actors did a poor job and the story made no sense"
]
sample_labels = [
    1, 1, 1, 1, 1, 1, 1, 1, 1, 1,  # First 10 are positive (1)
    0, 0, 0, 0, 0, 0, 0, 0, 0, 0   # Next 10 are negative (0)
]

# 1. Load and Preprocess Data
# Build vocabulary from all reviews
def tokenize(text):
    return text.lower().split()
all_tokens = []
for review in sample_reviews:
    all_tokens.extend(tokenize(review))
vocab = set(all_tokens)


def train_naive_bayes(texts, labels):
    """
    Train a Naive Bayes classifier.

    Returns:
        priors: dict mapping class to prior probability
        conditionals: dict mapping (word, class) to conditional probability P(word|class)
        vocab: set of all words in the vocabulary
    """
    num_docs = len(texts)
    class_counts = Counter(labels)
    priors = {c: class_counts[c] / num_docs for c in class_counts}

    word_counts_per_class = defaultdict(Counter)
    total_words_per_class = defaultdict(int)

    for text, label in zip(texts, labels):
        tokens = tokenize(text)
        word_counts_per_class[label].update(tokens)
        total_words_per_class[label] += len(tokens)

    vocab = set()
    for text in texts:
        vocab.update(tokenize(text))
    V = len(vocab)

    conditionals = {}
    for c in class_counts.keys():
        total_count_c = total_words_per_class[c]
        for w in vocab:
            count_wc = word_counts_per_class[c][w]
            # Laplace smoothing: (count + 1) / (total + |V|)
            prob = (count_wc + 1) / (total_count_c + V)
            conditionals[(w, c)] = prob

    return priors, conditionals, vocab


def predict_naive_bayes(text, priors, conditionals, vocab):
    """
    Predict the class for a given text using Naive Bayes.

    Returns:
        predicted_class: int (0 or 1)
        probabilities: dict mapping class to posterior probability
    """
    tokens = tokenize(text)

    log_posteriors = {}
    classes = list(priors.keys())
    for c in classes:

        log_p = math.log(priors[c])
        for w in tokens:
            if w in vocab:
                pwc = conditionals[(w, c)]
            else:
                pwc = 1e-8
            log_p += math.log(pwc)
        log_posteriors[c] = log_p

    max_log = max(log_posteriors.values())
    exp_vals = {c: math.exp(log_posteriors[c] - max_log) for c in classes}
    Z = sum(exp_vals.values())
    probs = {c: exp_vals[c] / Z for c in classes}

    predicted_class = max(probs, key=probs.get)
    return predicted_class, probs


# Train the model
priors, conditionals, vocab = train_naive_bayes(sample_reviews, sample_labels)

# Evaluate on training data
print("--- Task 1 Output ---")
for review, label in zip(sample_reviews, sample_labels):
    pred_class, probs = predict_naive_bayes(review, priors, conditionals, vocab)
    print(f"Review: {review}")
    print(f"True label: {label}, Predicted: {pred_class}")
    print(f"Probabilities: {probs}")
    print()

--- Task 1 Output ---
Review: this movie was fantastic and i really loved it
True label: 1, Predicted: 1
Probabilities: {1: 0.9589688293144206, 0: 0.04103117068557935}

Review: an amazing performance by the main actor
True label: 1, Predicted: 1
Probabilities: {1: 0.9916501809069962, 0: 0.008349819093003787}

Review: wonderful storyline and great cinematography
True label: 1, Predicted: 1
Probabilities: {1: 0.9490480359933321, 0: 0.05095196400666789}

Review: i enjoyed every minute of this film
True label: 1, Predicted: 1
Probabilities: {1: 0.946220741813882, 0: 0.05377925818611789}

Review: the film was outstanding and very entertaining
True label: 1, Predicted: 1
Probabilities: {1: 0.9488667079895284, 0: 0.05113329201047165}

Review: incredible acting and a moving plot
True label: 1, Predicted: 1
Probabilities: {1: 0.7619162458526316, 0: 0.23808375414736846}

Review: the best movie I have seen this year
True label: 1, Predicted: 1
Probabilities: {1: 0.9819535992008135, 0: 0.018046400

### Task 2: Implementing Viterbi Algorithm for HMM Sequence Tagging (15 Marks)

**Objective:** To implement the Viterbi algorithm for Hidden Markov Model (HMM) based sequence tagging, which is fundamental for tasks like POS tagging and named entity recognition.

**Description:** HMMs model sequences by assuming that each observation depends on a hidden state, and states follow a Markov chain. The Viterbi algorithm finds the most likely sequence of hidden states given observations.

**Your task is to:**

1.  **Define HMM Parameters:** Use the provided initial probabilities (π), transition probabilities (A), and emission probabilities (B) for a simple POS tagging task.
2.  **Implement Viterbi Algorithm:** Create a function `viterbi(observations, pi, A, B, states)` that:
    * Takes a sequence of observations (words), initial probabilities, transition matrix, emission matrix, and state labels.
    * Implements the Viterbi algorithm in log-space to find the most likely state sequence.
    * Returns the predicted state sequence and the log-probability of that sequence.
3.  **Test on Example Sentences:** Apply your Viterbi implementation to two sentences with the ambiguous word "book" and print the predicted tag sequences.
4.  **Analyze Results:** In a markdown cell, explain how the HMM resolves the ambiguity of "book" differently in the two sentences.

**Parameters:**

```python
# States (POS tags)
states = ['DET', 'NOUN', 'VERB', 'PRT']

# Initial probabilities (π)
pi = {'DET': 0.50, 'NOUN': 0.20, 'VERB': 0.20, 'PRT': 0.10}

# Transition probabilities (A): P(state_j | state_i)
A = {
    'DET': {'DET': 0.05, 'NOUN': 0.75, 'VERB': 0.15, 'PRT': 0.05},
    'NOUN': {'DET': 0.05, 'NOUN': 0.10, 'VERB': 0.75, 'PRT': 0.10},
    'VERB': {'DET': 0.10, 'NOUN': 0.35, 'VERB': 0.40, 'PRT': 0.15},
    'PRT': {'DET': 0.05, 'NOUN': 0.10, 'VERB': 0.75, 'PRT': 0.10}
}

# Emission probabilities (B): P(word | state)
B = {
    'DET': {'the': 0.80, 'a': 0.20},
    'NOUN': {'book': 0.45, 'table': 0.25, 'flight': 0.20, 'i': 0.05, 'on': 0.05},
    'VERB': {'is': 0.40, 'want': 0.35, 'book': 0.20, 'to': 0.03, 'on': 0.02},
    'PRT': {'to': 0.70, 'on': 0.30}
}

# Test sentences
sentence1 = ['the', 'book', 'is', 'on', 'the', 'table']
sentence2 = ['i', 'want', 'to', 'book', 'a', 'flight']
```

In [2]:
# Your code for Task 2 here
import math
import numpy as np
from typing import List, Tuple

# 1. Define HMM Parameters
states = ['DET', 'NOUN', 'VERB', 'PRT']

pi = {'DET': 0.50, 'NOUN': 0.20, 'VERB': 0.20, 'PRT': 0.10}

A = {
    'DET': {'DET': 0.05, 'NOUN': 0.75, 'VERB': 0.15, 'PRT': 0.05},
    'NOUN': {'DET': 0.05, 'NOUN': 0.10, 'VERB': 0.75, 'PRT': 0.10},
    'VERB': {'DET': 0.10, 'NOUN': 0.35, 'VERB': 0.40, 'PRT': 0.15},
    'PRT': {'DET': 0.05, 'NOUN': 0.10, 'VERB': 0.75, 'PRT': 0.10}
}

B = {
    'DET': {'the': 0.80, 'a': 0.20},
    'NOUN': {'book': 0.45, 'table': 0.25, 'flight': 0.20, 'i': 0.05, 'on': 0.05},
    'VERB': {'is': 0.40, 'want': 0.35, 'book': 0.20, 'to': 0.03, 'on': 0.02},
    'PRT': {'to': 0.70, 'on': 0.30}
}

# Handle unseen words (use a small probability)
UNK_PROB = 1e-8

def get_emission_prob(state, word):
    """Get emission probability P(word|state), handling unseen words."""
    if word in B.get(state, {}):
        return B[state][word]
    else:
        return UNK_PROB

# 2. Implement Viterbi Algorithm
def viterbi(observations, pi, A, B, states):
    """
    Viterbi algorithm to find the most likely state sequence.

    Args:
        observations: List of words (observations)
        pi: Initial state probabilities
        A: Transition probabilities A[state_i][state_j] = P(state_j | state_i)
        B: Emission probabilities B[state][word] = P(word | state)
        states: List of possible states

    Returns:
        best_path: List of states (most likely sequence)
        log_prob: Log probability of the best path
    """
    T = len(observations)
    N = len(states)
    v = [[-math.inf] * N for _ in range(T)]
    bp = [[None] * N for _ in range(T)]

    state_to_idx = {s: i for i, s in enumerate(states)}
    idx_to_state = {i: s for s, i in state_to_idx.items()}

    first_obs = observations[0]
    for s_idx, s in enumerate(states):
        p_init = pi.get(s, 0.0)
        if p_init == 0:
            v[0][s_idx] = -math.inf
            bp[0][s_idx] = None
        else:
            emit_p = get_emission_prob(s, first_obs)
            v[0][s_idx] = math.log(p_init) + math.log(emit_p)
            bp[0][s_idx] = None

    for t in range(1, T):
        obs = observations[t]
        for j, s_j in enumerate(states):
            emit_p = get_emission_prob(s_j, obs)
            log_emit = math.log(emit_p)
            best_prev_log_prob = -math.inf
            best_prev_state_idx = None
            for i, s_i in enumerate(states):
                trans_p = A[s_i].get(s_j, 0.0)
                if trans_p == 0:
                    continue
                log_prob = v[t-1][i] + math.log(trans_p)
                if log_prob > best_prev_log_prob:
                    best_prev_log_prob = log_prob
                    best_prev_state_idx = i
            v[t][j] = best_prev_log_prob + log_emit
            bp[t][j] = best_prev_state_idx

    last_t = T - 1
    best_last_state_idx = max(range(N), key=lambda i: v[last_t][i])
    best_log_prob = v[last_t][best_last_state_idx]

    best_path_indices = [best_last_state_idx]
    for t in range(last_t, 0, -1):
        best_last_state_idx = bp[t][best_last_state_idx]
        best_path_indices.append(best_last_state_idx)
    best_path_indices.reverse()

    best_path = [idx_to_state[i] for i in best_path_indices]
    return best_path, best_log_prob

# 3. Test on Example Sentences
sentence1 = ['the', 'new', 'book', 'is', 'on', 'the', 'table'] # 'new' is an OOV word
sentence2 = ['i', 'want', 'to', 'book', 'a', 'flight']

# Run Viterbi on both sentences
path1, logp1 = viterbi(sentence1, pi, A, B, states)
path2, logp2 = viterbi(sentence2, pi, A, B, states)

# Print results
print("--- Task 2 Output ---")
print(f"Sentence 1: {sentence1}")
print(f"Predicted tags: {list(zip(sentence1, path1))}")
print(f"Log probability: {logp1:.3f}\n")

print(f"Sentence 2: {sentence2}")
print(f"Predicted tags: {list(zip(sentence2, path2))}")
print(f"Log probability: {logp2:.3f}")

--- Task 2 Output ---
Sentence 1: ['the', 'new', 'book', 'is', 'on', 'the', 'table']
Predicted tags: [('the', 'DET'), ('new', 'NOUN'), ('book', 'VERB'), ('is', 'VERB'), ('on', 'PRT'), ('the', 'DET'), ('table', 'NOUN')]
Log probability: -31.348

Sentence 2: ['i', 'want', 'to', 'book', 'a', 'flight']
Predicted tags: [('i', 'NOUN'), ('want', 'VERB'), ('to', 'PRT'), ('book', 'VERB'), ('a', 'DET'), ('flight', 'NOUN')]
Log probability: -15.903


### Analysis for Task 2

In this task, we apply the Viterbi algorithm in an HMM for POS tagging on two sentences containing the ambiguous word "book":

1. Sentence 1: `the new book is on the table`  
2. Sentence 2: `i want to book a flight`

The word "book" can be either a NOUN ("a book") or a VERB ("to book a flight"). The HMM resolves this ambiguity by combining:

1. Transition probabilities (A): how likely one tag follows another  
2. Emission probabilities (B): how likely a word is generated by a tag  

Viterbi finds the most probable tag sequence in log-space.



### Sentence 1: "the new book is on the table"

Here, "book" is tagged as a NOUN:

```python
# High transition from DET to NOUN
A['DET']['NOUN'] = 0.75

# Higher emission as noun than verb
B['NOUN']['book'] = 0.45
B['VERB']['book'] = 0.20

# Probable NOUN → VERB pattern after "book"
A['NOUN']['VERB'] = 0.75
B['VERB']['is'] = 0.40
```

The context `DET ... book is` plus strong transitions lead to "book" as NOUN.


### Sentence 2: "i want to book a flight"

Here, "book" is tagged as a VERB:

```python
# VERB → PRT → VERB pattern
B['VERB']['want'] = 0.35
B['PRT']['to'] = 0.70
A['VERB']['PRT'] = 0.15
A['PRT']['VERB'] = 0.75

# Weak transition from PRT to NOUN
A['PRT']['NOUN'] = 0.10

# Following words fit VERB → DET → NOUN
B['NOUN']['flight'] = 0.20
```

The context `want to book a flight` plus strong PRT → VERB transition makes "book" as VERB more probable.


### Summary

1. Sentence 1: Context `DET ... book is` with high DET → NOUN and NOUN → VERB transitions tags "book" as NOUN.
2. Sentence 2: Pattern VERB → PRT → VERB → DET → NOUN and the strong PRT → VERB transition tags "book" as VERB.

The HMM uses both transition and emission probabilities; Viterbi finds the globally highest-probability sequence, allowing "book" to be disambiguated differently based on context.

### Task 3: Conditional Random Fields (CRF) for Sequence Labeling (15 Marks)

**Objective:** To understand Conditional Random Fields (CRFs), which extend HMMs by modeling global dependencies and avoiding the independence assumptions of HMMs.

**Description:** CRFs are discriminative models that directly model the conditional probability P(tags|words) rather than the joint probability. Unlike HMMs, CRFs can incorporate features from the entire sequence and avoid the Markov independence assumption.

**Your task is to:**

1.  **Understand CRF Features:** CRFs use feature functions.
2.  **Use CRF Library:** Use the `sklearn-crfsuite` library (or `pytorch-crf`) to train a CRF model for NER (Named Entity Recognition) on a small dataset.
3.  **Feature Extraction:** Create a function `extract_features(sentence, i)` that extracts features for word at position i, including:
    * The word itself
    * Previous word (if available)
    * Next word (if available)
    * Word shape (capitalization pattern)
    * Word prefix/suffix (first/last 2-3 characters)
4.  **Train and Evaluate:** Train a CRF model on the provided training data and evaluate on test sentences.

**Dataset:**

```python
# Simple NER dataset
train_sentences = [
    (['Barack', 'Obama', 'was', 'born', 'in', 'Hawaii'], ['B-PER', 'I-PER', 'O', 'O', 'O', 'B-LOC']),
    (['Apple', 'is', 'based', 'in', 'California'], ['B-ORG', 'O', 'O', 'O', 'B-LOC']),
    (['Microsoft', 'headquarters', 'is', 'in', 'Seattle'], ['B-ORG', 'O', 'O', 'O', 'B-LOC'])
]

test_sentences = [
    (['Joe', 'Biden', 'visited', 'New', 'York'], ['B-PER', 'I-PER', 'O', 'B-LOC', 'I-LOC']),
    (['Amazon', 'is', 'in', 'Washington'], ['B-ORG', 'O', 'O', 'B-LOC'])
]
```

**Note:** You may need to install `sklearn-crfsuite` using `pip install sklearn-crfsuite`.

In [3]:
# Your code for Task 3 here
!pip install sklearn-crfsuite -q

import sklearn_crfsuite
from sklearn_crfsuite import CRF

# Dataset
train_sentences = [
    (['Barack', 'Obama', 'was', 'born', 'in', 'Hawaii'], ['B-PER', 'I-PER', 'O', 'O', 'O', 'B-LOC']),
    (['Apple', 'is', 'based', 'in', 'California'], ['B-ORG', 'O', 'O', 'O', 'B-LOC']),
    (['Microsoft', 'headquarters', 'is', 'in', 'Seattle'], ['B-ORG', 'O', 'O', 'O', 'B-LOC'])
]

test_sentences = [
    (['Joe', 'Biden', 'visited', 'New', 'York'], ['B-PER', 'I-PER', 'O', 'B-LOC', 'I-LOC']),
    (['Amazon', 'is', 'in', 'Washington'], ['B-ORG', 'O', 'O', 'B-LOC'])
]

def word2features(sentence, i):
    """
    Extract features for word at position i in the sentence.

    Returns:
        dict: Dictionary of features
    """
    word = sentence[i]
    features = {
        'word': word.lower(),
        'word_upper': word.isupper(),
        'word_title': word.istitle(),
        'word_isdigit': word.isdigit(),
    }

    # Add features for previous and next word, prefixes, suffixes, and word shape
    if i > 0:
        prev_word = sentence[i-1]
        features.update({
            'prev_word': prev_word.lower(),
            'prev_word_is_title': prev_word.istitle(),
            'prev_word_is_upper': prev_word.isupper(),
        })
    else:
        features['BOS'] = True  # Beginning of sentence

    if i < len(sentence) - 1:
        next_word = sentence[i+1]
        features.update({
            'next_word': next_word.lower(),
            'next_word_is_title': next_word.istitle(),
            'next_word_is_upper': next_word.isupper(),
        })
    else:
        features['EOS'] = True  # End of sentence

    # Word prefixes and suffixes
    features['prefix2'] = word[:2]
    features['prefix3'] = word[:3]
    features['suffix2'] = word[-2:]
    features['suffix3'] = word[-3:]

    # Word shape (simple version)
    def word_shape(w):
        shape = ''
        for c in w:
            if c.isupper():
                shape += 'X'
            elif c.islower():
                shape += 'x'
            elif c.isdigit():
                shape += 'd'
            else:
                shape += c
        return shape
    features['word_shape'] = word_shape(word)

    return features

def sentence2features(sentence):
    """Convert a sentence to a list of feature dictionaries."""
    return [word2features(sentence, i) for i in range(len(sentence))]

def sentence2labels(sentence_labels):
    """Convert labels to list format."""
    return list(sentence_labels)

# Prepare training data
X_train = [sentence2features(sent) for sent, labels in train_sentences]
y_train = [sentence2labels(labels) for sent, labels in train_sentences]

# Prepare test data
X_test = [sentence2features(sent) for sent, labels in test_sentences]
y_test = [sentence2labels(labels) for sent, labels in test_sentences]

# Train CRF model
crf = CRF(algorithm='lbfgs', c1=0.1, c2=0.1, max_iterations=100, all_possible_transitions=True)
crf.fit(X_train, y_train)

# Evaluate
print("--- Task 3 Output ---")
for i, (sent, true_labels) in enumerate(test_sentences):
    X_sent = [sentence2features(sent)]
    predicted = crf.predict(X_sent)[0]
    print(f"Sentence: {sent}")
    print(f"True labels: {true_labels}")
    print(f"Predicted: {predicted}")
    print()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 23.1 MB/s eta 0:00:00
--- Task 3 Output ---
Sentence: ['Joe', 'Biden', 'visited', 'New', 'York']
True labels: ['B-PER', 'I-PER', 'O', 'B-LOC', 'I-LOC']
Predicted: ['B-ORG' 'O' 'O' 'O' 'B-LOC']

Sentence: ['Amazon', 'is', 'in', 'Washington']
True labels: ['B-ORG', 'O', 'O', 'B-LOC']
Predicted: ['B-ORG' 'O' 'O' 'B-LOC']



### Task 4: Sequence Classification with LSTM (20 Marks)

**Objective:** To implement a sequence classification model using LSTM networks to understand how recurrent neural networks process sequential data.

**Description:** LSTMs are neural architectures designed to process sequences by maintaining hidden states that capture information from previous time steps. LSTMs use gating mechanisms (forget, input, and output gates) to better handle long-term dependencies compared to basic RNNs.

**Your task is to:**

1.  **Load and Preprocess Data:**
    * The IMDB dataset loading and splitting is provided for you (using `torchtext`).
    * Tokenize the text, build a vocabulary from the training set, and convert sequences to integer indices.
    * Add special tokens: `<PAD>` (index 0) for padding and `<UNK>` (index 1) for unknown words.
2.  **Pad Sequences:** Implement a function to pad or truncate all sequences to a fixed length (e.g., `max_length = 50`).
3.  **Define LSTM Model:** Create a PyTorch `nn.Module` class `LSTMClassifier` with:
    * An `nn.Embedding` layer.
    * An `nn.LSTM` layer (use `batch_first=True`).
    * A final `nn.Linear` layer for classification.
4.  **Train the Model:**
    * Define a loss function (e.g., `nn.BCEWithLogitsLoss` for binary classification).
    * Create an optimizer (e.g., `torch.optim.Adam` with learning rate 0.001).
    * Implement a training loop that:
      - Iterates through the data in batches (use `batch_size = 32`).
      - Performs forward passes for the LSTM model.
      - Computes the loss.
      - Performs backpropagation (zero gradients, backward, step optimizer).
      - Updates model parameters.
    * Train for multiple epochs (e.g., 10 epochs) and print the loss periodically.
5.  **Evaluate:** After training, evaluate the model on the validation set:
    * Set model to evaluation mode.
    * Compute predictions on validation data.
    * Calculate and print validation accuracy.
    * Show some example predictions with their true labels.

**Dataset:**

You will use the **IMDB Movie Reviews Dataset** for sentiment classification. This is a binary classification dataset where reviews are labeled as positive (1) or negative (0). The dataset will be loaded using `torchtext.datasets.IMDB` and automatically split into training (80%) and validation (20%) sets. A subset of 2000 samples will be used to keep training time reasonable.

**Note:** The data loading and splitting code is provided for you. You need to focus on building the vocabulary, implementing the models, and training them.

**Parameters:**

```python
embed_dim = 64
hidden_dim = 128
output_dim = 1   # Binary classification
max_length = 50
```

In [18]:
# ==== Task 4: Sentiment Classification with LSTM (Fixed Path Logic) ====
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import random
import numpy as np
from collections import Counter
import tarfile
import requests
import os
import re

# 1. Setup Seeds
SEED = 1234
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ==============================================================================
# 2. Manual Data Loading (Bypassing broken torchtext)
# ==============================================================================
def download_and_extract_imdb():
    url = "http://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz"
    filename = "aclImdb_v1.tar.gz"
    dirname = "aclImdb"

    if not os.path.exists(dirname):
        print("Downloading IMDB dataset (this may take a moment)...")
        if not os.path.exists(filename):
            try:
                response = requests.get(url, stream=True)
                with open(filename, "wb") as f:
                    for chunk in response.iter_content(chunk_size=1024):
                        if chunk:
                            f.write(chunk)
            except Exception as e:
                print(f"Download failed: {e}")
                return None

        print("Extracting IMDB dataset...")
        try:
            with tarfile.open(filename, "r:gz") as tar:
                tar.extractall()
        except Exception as e:
            print(f"Extraction failed: {e}")
            return None

    return dirname

def load_imdb_data(root_dir, split='train'):
    data = []
    # IMDB structure: aclImdb/train/pos, aclImdb/train/neg
    split_dir = os.path.join(root_dir, split)

    if not os.path.exists(split_dir):
        raise FileNotFoundError(f"Directory not found: {split_dir}. Check if extraction worked correctly.")

    for label in ['pos', 'neg']:
        label_dir = os.path.join(split_dir, label)
        sentiment = 1.0 if label == 'pos' else 0.0

        if not os.path.exists(label_dir):
             print(f"Warning: {label_dir} not found, skipping.")
             continue

        for filename in os.listdir(label_dir):
            if filename.endswith(".txt"):
                with open(os.path.join(label_dir, filename), 'r', encoding='utf-8') as f:
                    text = f.read()
                    data.append((text, sentiment))
    return data

# Execute download and load
data_root = download_and_extract_imdb()

if data_root is None:
    raise RuntimeError("Failed to download or extract dataset.")

print(f"Data root is: {data_root}")
print("Loading raw text data...")

# FIX: Do not append "aclImdb" again. The tar extracts directly into 'aclImdb' folder.
full_train_data = load_imdb_data(data_root, split='train')

# Shuffle BEFORE slicing to ensure mixed labels
random.shuffle(full_train_data)

# Slice to 2000 samples for the assignment
MAX_SAMPLES = 2000
if len(full_train_data) == 0:
    raise ValueError("No data loaded. Check directory structure.")

subset_data = full_train_data[:MAX_SAMPLES]

texts = [item[0] for item in subset_data]
labels = [item[1] for item in subset_data]

# Split into Train (80%) and Validation (20%)
split_idx = int(len(texts) * 0.8)
train_texts = texts[:split_idx]
train_labels = labels[:split_idx]
val_texts = texts[split_idx:]
val_labels = labels[split_idx:]

print(f"Training samples: {len(train_texts)}")
print(f"Validation samples: {len(val_texts)}")
print(f"Train label mean: {np.mean(train_labels):.2f}")

# ==============================================================================
# 3. Preprocessing & Tokenization
# ==============================================================================
# Simple tokenizer using regex
def simple_tokenizer(text):
    # Convert to lowercase and split by non-alphanumeric characters
    return re.findall(r'\b\w+\b', text.lower())

# Build Vocabulary
print("Building vocabulary...")
token_counts = Counter()
for text in train_texts:
    token_counts.update(simple_tokenizer(text))

vocab = {"<PAD>": 0, "<UNK>": 1}
min_freq = 2
for token, freq in token_counts.items():
    if freq >= min_freq:
        vocab[token] = len(vocab)

print(f"Vocab size: {len(vocab)}")

def text_pipeline(text):
    return [vocab.get(token, vocab["<UNK>"]) for token in simple_tokenizer(text)]

# ==============================================================================
# 4. DataLoader & Collate
# ==============================================================================
max_len = 200

def collate_batch(batch):
    label_list, text_list = [], []
    for (_text, _label) in batch:
        label_list.append(_label)
        processed_text = text_pipeline(_text)

        if len(processed_text) > max_len:
            processed_text = processed_text[:max_len]
        else:
            processed_text = processed_text + [vocab["<PAD>"]] * (max_len - len(processed_text))

        text_list.append(processed_text)

    label_list = torch.tensor(label_list, dtype=torch.float32).unsqueeze(1)
    text_list = torch.tensor(text_list, dtype=torch.long)

    return text_list.to(device), label_list.to(device)

train_data_loader_input = list(zip(train_texts, train_labels))
val_data_loader_input = list(zip(val_texts, val_labels))

BATCH_SIZE = 32
train_loader = DataLoader(train_data_loader_input, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_batch)
val_loader = DataLoader(val_data_loader_input, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_batch)

# ==============================================================================
# 5. Model Definition (LSTM)
# ==============================================================================
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, output_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=vocab["<PAD>"])
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)
        self.dropout = nn.Dropout(0.5)

    def forward(self, text):
        embedded = self.embedding(text)
        output, (hidden, cell) = self.lstm(embedded)
        last_hidden = hidden[-1, :, :]
        return self.fc(self.dropout(last_hidden))

EMBED_DIM = 64
HIDDEN_DIM = 128
OUTPUT_DIM = 1
LEARNING_RATE = 0.001
NUM_EPOCHS = 5

model = LSTMClassifier(len(vocab), EMBED_DIM, HIDDEN_DIM, OUTPUT_DIM)
model = model.to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

# ==============================================================================
# 6. Training Loop
# ==============================================================================
def binary_accuracy(preds, y):
    rounded_preds = torch.round(torch.sigmoid(preds))
    correct = (rounded_preds == y).float()
    acc = correct.sum() / len(correct)
    return acc

print("Start Training...")
for epoch in range(NUM_EPOCHS):
    model.train()
    epoch_loss = 0
    epoch_acc = 0

    for text, label in train_loader:
        optimizer.zero_grad()
        predictions = model(text)
        loss = criterion(predictions, label)
        acc = binary_accuracy(predictions, label)

        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        epoch_acc += acc.item()

    train_loss = epoch_loss / len(train_loader)
    train_acc = epoch_acc / len(train_loader)

    model.eval()
    val_loss = 0
    val_acc = 0
    with torch.no_grad():
        for text, label in val_loader:
            predictions = model(text)
            loss = criterion(predictions, label)
            acc = binary_accuracy(predictions, label)
            val_loss += loss.item()
            val_acc += acc.item()

    val_loss = val_loss / len(val_loader)
    val_acc = val_acc / len(val_loader)

    print(f'Epoch: {epoch+1:02} | Train Loss: {train_loss:.3f} | Train Acc: {train_acc*100:.2f}% | Val Loss: {val_loss:.3f} | Val Acc: {val_acc*100:.2f}%')

print("Task 4 Completed.")

Using device: cuda
Data root is: aclImdb
Loading raw text data...
Training samples: 1600
Validation samples: 400
Train label mean: 0.51
Building vocabulary...
Vocab size: 12524
Start Training...
Epoch: 01 | Train Loss: 0.693 | Train Acc: 50.56% | Val Loss: 0.694 | Val Acc: 46.88%
Epoch: 02 | Train Loss: 0.683 | Train Acc: 55.88% | Val Loss: 0.704 | Val Acc: 46.88%
Epoch: 03 | Train Loss: 0.667 | Train Acc: 58.81% | Val Loss: 0.706 | Val Acc: 47.12%
Epoch: 04 | Train Loss: 0.636 | Train Acc: 60.62% | Val Loss: 0.719 | Val Acc: 46.63%
Epoch: 05 | Train Loss: 0.635 | Train Acc: 61.19% | Val Loss: 0.733 | Val Acc: 46.63%
Task 4 Completed.


### Task 5: Implementing Attention Mechanism (from scratch) (15 Marks)

**Objective:** To understand the attention mechanism by implementing basic dot-product attention and scaled dot-product attention from scratch.

**Description:** Attention allows models to focus on different parts of the input sequence when making predictions. This is a key innovation that led to the Transformer architecture. You will implement both a basic dot-product attention and the scaled dot-product attention used in Transformers.

**Your task is to:**

1.  **Implement Basic Dot-Product Attention:**
    * Use the provided `encoder_hidden_states` and `decoder_hidden_state`.
    * Calculate alignment scores using dot-product: `score = decoder_hidden_state @ encoder_hidden_states.T`.
    * Apply softmax to get attention weights.
    * Compute context vector as weighted sum of encoder states.
    * Print attention weights and context vector.

2.  **Implement Scaled Dot-Product Attention:**
    * Implement function `scaled_dot_product_attention(Q, K, V, mask=None)`.
    * Calculate scores: `scores = Q @ K.T / sqrt(d_k)`.
    * Apply mask if provided (set masked positions to -1e9 before softmax).
    * Apply softmax and multiply by V.
    * Return output and attention weights.



In [7]:
# Your code for Task 5 here
import numpy as np
import torch
import torch.nn.functional as F
import math

# Parameters for Task 5
np.random.seed(42)
torch.manual_seed(42)

# For basic attention
T_seq_len = 5
h_dim = 64
encoder_hidden_states = np.random.rand(T_seq_len, h_dim)
decoder_hidden_state = np.random.rand(1, h_dim)

# For scaled dot-product attention
batch_size = 1
seq_len = 4
d_k = 64
Q = torch.rand(batch_size, seq_len, d_k)
K = torch.rand(batch_size, seq_len, d_k)
V = torch.rand(batch_size, seq_len, d_k)
mask = torch.tril(torch.ones(seq_len, seq_len)).unsqueeze(0) # Causal mask for decoder

# 1. Implement Basic Dot-Product Attention
def basic_attention(decoder_hidden, encoder_hidden_states):
    """
    Compute basic attention weights and context vector.

    Args:
        decoder_hidden: (1, h_dim) decoder hidden state
        encoder_hidden_states: (T_seq_len, h_dim) encoder hidden states

    Returns:
        attention_weights: (1, T_seq_len) attention weights
        context_vector: (1, h_dim) context vector
    """
    # Calculate alignment scores using dot-product: (1, h_dim) x (h_dim, T) -> (1, T)
    scores = np.dot(decoder_hidden, encoder_hidden_states.T)  # shape (1, T_seq_len)
    # Apply softmax to get attention weights
    scores_shifted = scores - np.max(scores)  # numerical stability
    exp_scores = np.exp(scores_shifted)
    attention_weights = exp_scores / np.sum(exp_scores, axis=1, keepdims=True)  # (1, T_seq_len)
    # Compute context vector as weighted sum
    context_vector = np.dot(attention_weights, encoder_hidden_states)  # (1, h_dim)
    return attention_weights, context_vector

# Run basic attention
attention_weights, context_vector = basic_attention(decoder_hidden_state, encoder_hidden_states)

# 2. Implement Scaled Dot-Product Attention
def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Scaled dot-product attention as used in Transformers.

    Args:
        Q: (batch_size, seq_len, d_k) Query matrix
        K: (batch_size, seq_len, d_k) Key matrix
        V: (batch_size, seq_len, d_k) Value matrix
        mask: (batch_size, seq_len, seq_len) optional mask (0 for masked positions)

    Returns:
        output: (batch_size, seq_len, d_k) attention output
        attention_weights: (batch_size, seq_len, seq_len) attention weights
    """
    # Calculate d_k from K
    d_k = K.size(-1)
    # Compute scores = Q @ K.T / sqrt(d_k)
    # K.transpose(1,2): (batch, d_k, seq_len)
    scores = torch.matmul(Q, K.transpose(1, 2)) / math.sqrt(d_k)  # (batch, seq_len, seq_len)

    # Apply mask if provided (set masked positions to -1e9)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, -1e9)

    # Apply softmax
    attention_weights = F.softmax(scores, dim=-1)  # (batch, seq_len, seq_len)

    # Multiply by V
    output = torch.matmul(attention_weights, V)  # (batch, seq_len, d_k)

    return output, attention_weights

# Run scaled dot-product attention (without mask)
output, weights = scaled_dot_product_attention(Q, K, V)

# Run scaled dot-product attention (with mask)
output_masked, weights_masked = scaled_dot_product_attention(Q, K, V, mask=mask)


# 3. Compare Results
print("--- Task 5 Output ---")
print("Basic Dot-Product Attention:")
print(f"Attention weights shape: {attention_weights.shape}")
print(f"Context vector shape: {context_vector.shape}")
print(f"Attention weights: {attention_weights}")
print()
print("Scaled Dot-Product Attention (Without Mask):")
print(f"Output shape: {output.shape}")
print(f"Attention weights shape: {weights.shape}")
print(f"Attention weights (first sequence):\n{weights[0]}")
print()
print("Scaled Dot-Product Attention (With Causal Mask):")
print(f"Output shape: {output_masked.shape}")
print(f"Attention weights shape: {weights_masked.shape}")
print(f"Attention weights (first sequence):\n{weights_masked[0]}")

--- Task 5 Output ---
Basic Dot-Product Attention:
Attention weights shape: (1, 5)
Context vector shape: (1, 64)
Attention weights: [[0.05497059 0.16505629 0.06604092 0.52198085 0.19195135]]

Scaled Dot-Product Attention (Without Mask):
Output shape: torch.Size([1, 4, 64])
Attention weights shape: torch.Size([1, 4, 4])
Attention weights (first sequence):
tensor([[0.2394, 0.2234, 0.2553, 0.2819],
        [0.2566, 0.2076, 0.2514, 0.2845],
        [0.2524, 0.2667, 0.2302, 0.2507],
        [0.2463, 0.2153, 0.2533, 0.2851]])

Scaled Dot-Product Attention (With Causal Mask):
Output shape: torch.Size([1, 4, 64])
Attention weights shape: torch.Size([1, 4, 4])
Attention weights (first sequence):
tensor([[1.0000, 0.0000, 0.0000, 0.0000],
        [0.5528, 0.4472, 0.0000, 0.0000],
        [0.3368, 0.3559, 0.3073, 0.0000],
        [0.2463, 0.2153, 0.2533, 0.2851]])


### Task 6: Using BERT as Feature Extractor with MLP Classifier (20 Marks)

**Objective:** To use a pre-trained BERT model as a feature extractor and train a separate MLP classifier on top of BERT's features.

**Description:** Instead of fine-tuning BERT end-to-end, we can use BERT as a frozen feature extractor. This approach extracts contextualized embeddings from BERT and trains a separate classifier (MLP) on these features. This is computationally more efficient than fine-tuning and can be effective when you have limited training data or computational resources.

**Your task is to:**

1.  **Load Pre-trained BERT:**
    * Install the `transformers` library if needed.
    * Import `BertTokenizer` and `BertModel`.
    * Load the pre-trained model and tokenizer for `"bert-base-uncased"`.
    * Freeze all BERT parameters (set `requires_grad=False` for all parameters).

2.  **Prepare Data:**
    * Use the IMDB dataset from Task 4 (or a subset of it).
    * Tokenize the reviews using the BERT tokenizer with `padding=True`, `truncation=True`, and `return_tensors='pt'`.
    * Split into training and validation sets (80-20 split).

3.  **Extract BERT Features:**
    * Pass tokenized inputs through BERT to get contextualized embeddings.
    * Use the `[CLS]` token embedding (first token) as the sentence representation.
    * Extract features for all training and validation samples (in batches).

4.  **Define and Train MLP Classifier:**
    * Create a Multi-Layer Perceptron (MLP) classifier with:
      * Input layer: size 768 (BERT hidden size)
      * One or more hidden layers with ReLU activation
      * Output layer: size 2 (for binary classification)
    * Define a loss function (`CrossEntropyLoss`).
    * Create an optimizer (e.g., `Adam` with learning rate 0.001) that only updates the MLP parameters.
    * Implement a training loop to train the MLP on the extracted BERT features.
    * Train for several epochs (e.g., 10 epochs) and print training loss periodically.

5.  **Evaluate:**
    * Evaluate the MLP on the validation set using the extracted BERT features.
    * Calculate and print validation accuracy.
    * Show some example predictions with their true labels.


In [21]:
# ==== Task 6: Using BERT as Feature Extractor with MLP Classifier ====
# Use BERT as a frozen feature extractor and train an MLP classifier on top

# !pip install transformers -q

import os
import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from transformers import BertTokenizer, BertModel

# Fix random seeds for reproducibility
SEED = 1234
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ============================================================
# 1. Load IMDB data from local aclImdb folder (reuse from Task 4)
# ============================================================

def load_imdb_from_folder(root_dir, split='train', max_samples=1000):
    """
    Load IMDB reviews and labels from a local 'aclImdb' folder.

    Args:
        root_dir (str): Root directory of IMDB data, e.g., 'aclImdb'.
        split (str): 'train' or 'test'.
        max_samples (int): Maximum number of samples to load.

    Returns:
        texts (List[str]): List of review texts.
        labels (List[int]): List of labels (1 = positive, 0 = negative).
    """
    texts = []
    labels = []
    split_dir = os.path.join(root_dir, split)

    for label_name in ['pos', 'neg']:
        label_dir = os.path.join(split_dir, label_name)
        if not os.path.exists(label_dir):
            raise FileNotFoundError(
                f"{label_dir} not found. Please check that the 'aclImdb' dataset exists in the correct path."
            )

        sentiment = 1 if label_name == 'pos' else 0
        for fname in os.listdir(label_dir):
            if fname.endswith(".txt"):
                with open(os.path.join(label_dir, fname), 'r', encoding='utf-8') as f:
                    text = f.read()
                texts.append(text)
                labels.append(sentiment)

    # Shuffle and limit to max_samples
    combined = list(zip(texts, labels))
    random.shuffle(combined)
    combined = combined[:max_samples]
    texts, labels = zip(*combined)
    return list(texts), list(labels)

# Load a subset (e.g., 1000 examples) from local IMDB (train split)
data_root = "aclImdb"
max_samples = 1000
texts, labels = load_imdb_from_folder(data_root, split='train', max_samples=max_samples)

print(f"Total loaded samples from IMDB(train): {len(texts)}")

# 80/20 train/validation split
train_texts, val_texts, train_labels, val_labels = train_test_split(
    texts, labels, test_size=0.2, random_state=SEED, stratify=labels
)

print(f"Training samples: {len(train_texts)}")
print(f"Validation samples: {len(val_texts)}")
print(f"Train label mean: {np.mean(train_labels):.2f}")

# ============================================================
# 2. Load pre-trained BERT and freeze its parameters
# ============================================================

# Load tokenizer and model
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
bert_model = BertModel.from_pretrained('bert-base-uncased')

# Freeze all BERT parameters (we only train the MLP on top)
for param in bert_model.parameters():
    param.requires_grad = False

bert_model.to(device)
bert_model.eval()
print("BERT model loaded and frozen.")

# ============================================================
# 3. Encode texts with BERT tokenizer and extract [CLS] features
# ============================================================

def encode_texts(text_list, tokenizer, max_len=128, batch_size=16):
    """
    Encode a list of texts using the BERT tokenizer and extract [CLS] embeddings
    by passing them through the frozen BERT model.

    Args:
        text_list (List[str]): List of raw text reviews.
        tokenizer: BERT tokenizer.
        max_len (int): Maximum sequence length for BERT.
        batch_size (int): Batch size for encoding.

    Returns:
        torch.Tensor of shape (num_samples, 768): BERT [CLS] embeddings.
    """
    all_features = []

    # Process in batches to avoid memory issues
    for i in range(0, len(text_list), batch_size):
        batch_texts = text_list[i:i+batch_size]
        encodings = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=max_len,
            return_tensors='pt'
        )
        input_ids = encodings['input_ids'].to(device)
        attention_mask = encodings['attention_mask'].to(device)

        with torch.no_grad():
            outputs = bert_model(input_ids=input_ids, attention_mask=attention_mask)
            # Use the [CLS] token representation (first token) as sentence embedding
            cls_embeddings = outputs.last_hidden_state[:, 0, :]  # (batch_size, 768)
        all_features.append(cls_embeddings.cpu())  # Move to CPU to save GPU memory

    all_features = torch.cat(all_features, dim=0)  # (N, 768)
    return all_features

print("Extracting BERT [CLS] features for training set...")
train_features = encode_texts(train_texts, tokenizer, max_len=128, batch_size=16)
train_labels_tensor = torch.tensor(train_labels, dtype=torch.long)

print("Extracting BERT [CLS] features for validation set...")
val_features = encode_texts(val_texts, tokenizer, max_len=128, batch_size=16)
val_labels_tensor = torch.tensor(val_labels, dtype=torch.long)

print(f"Training features shape: {train_features.shape}")
print(f"Validation features shape: {val_features.shape}")

# Create DataLoaders for training the MLP
mlp_train_dataset = TensorDataset(train_features, train_labels_tensor)
mlp_val_dataset = TensorDataset(val_features, val_labels_tensor)

mlp_train_loader = DataLoader(mlp_train_dataset, batch_size=32, shuffle=True)
mlp_val_loader = DataLoader(mlp_val_dataset, batch_size=32, shuffle=False)

# ============================================================
# 4. Define and train the MLP classifier (on top of BERT features)
# ============================================================

class MLPClassifier(nn.Module):
    def __init__(self, input_dim=768, hidden_dim=256, output_dim=2, dropout=0.1):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.act = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.fc2 = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        x = self.fc1(x)
        x = self.act(x)
        x = self.dropout(x)
        x = self.fc2(x)  # No softmax here; CrossEntropyLoss applies log-softmax internally
        return x

mlp_model = MLPClassifier(input_dim=768, hidden_dim=256, output_dim=2, dropout=0.1)
mlp_model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(mlp_model.parameters(), lr=1e-3)

num_epochs = 10

print("Start training MLP classifier on top of frozen BERT features...")
for epoch in range(1, num_epochs + 1):
    mlp_model.train()
    epoch_loss = 0.0
    correct = 0
    total = 0

    for features_batch, labels_batch in mlp_train_loader:
        features_batch = features_batch.to(device)
        labels_batch = labels_batch.to(device)

        optimizer.zero_grad()
        logits = mlp_model(features_batch)
        loss = criterion(logits, labels_batch)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item() * labels_batch.size(0)

        preds = torch.argmax(logits, dim=1)
        correct += (preds == labels_batch).sum().item()
        total += labels_batch.size(0)

    avg_loss = epoch_loss / total
    train_acc = correct / total

    print(
        f"Epoch {epoch:02d}/{num_epochs} | "
        f"Train Loss: {avg_loss:.4f} | Train Acc: {train_acc*100:.2f}%"
    )

# ============================================================
# 5. Evaluate MLP + BERT features on the validation set
# ============================================================

mlp_model.eval()
correct = 0
total = 0

with torch.no_grad():
    for features_batch, labels_batch in mlp_val_loader:
        features_batch = features_batch.to(device)
        labels_batch = labels_batch.to(device)

        logits = mlp_model(features_batch)
        preds = torch.argmax(logits, dim=1)

        correct += (preds == labels_batch).sum().item()
        total += labels_batch.size(0)

val_accuracy = correct / total if total > 0 else 0.0

print("--- Task 6 Output ---")
print(f"Validation Accuracy: {val_accuracy:.4f}")

# Show some example predictions on the validation set
print("\nSome example predictions on validation set:")
num_examples_to_show = min(5, len(val_texts))
with torch.no_grad():
    example_features = val_features[:num_examples_to_show].to(device)
    example_logits = mlp_model(example_features)
    example_preds = torch.argmax(example_logits, dim=1).cpu().tolist()

for i in range(num_examples_to_show):
    print(f"\nReview {i+1}: {val_texts[i][:120].replace('\\n', ' ')} ...")
    print(f"True label: {val_labels[i]}  (1=pos, 0=neg)")
    print(f"Predicted : {example_preds[i]}")

Using device: cuda
Total loaded samples from IMDB(train): 1000
Training samples: 800
Validation samples: 200
Train label mean: 0.52
BERT model loaded and frozen.
Extracting BERT [CLS] features for training set...
Extracting BERT [CLS] features for validation set...
Training features shape: torch.Size([800, 768])
Validation features shape: torch.Size([200, 768])
Start training MLP classifier on top of frozen BERT features...
Epoch 01/10 | Train Loss: 0.5938 | Train Acc: 70.00%
Epoch 02/10 | Train Loss: 0.4607 | Train Acc: 79.50%
Epoch 03/10 | Train Loss: 0.4404 | Train Acc: 80.25%
Epoch 04/10 | Train Loss: 0.4072 | Train Acc: 82.38%
Epoch 05/10 | Train Loss: 0.3672 | Train Acc: 83.75%
Epoch 06/10 | Train Loss: 0.3423 | Train Acc: 85.88%
Epoch 07/10 | Train Loss: 0.3340 | Train Acc: 84.88%
Epoch 08/10 | Train Loss: 0.3035 | Train Acc: 86.75%
Epoch 09/10 | Train Loss: 0.2735 | Train Acc: 88.38%
Epoch 10/10 | Train Loss: 0.2988 | Train Acc: 86.88%
--- Task 6 Output ---
Validation Accuracy: